In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import math
import copy
from utils import *
import matplotlib.pyplot as plt

from diffusion import *
from diff_joint import *
from data_loader import *
from combined_data_loader import *
from Optimization_multi_node import *
from tqdm import tqdm

In [35]:

class Args:
    def __init__(self):
        self.T = 24
        self.base_mva = 100.0
        self.capacity_scale = 8
        self.ramp_rate = 0.5
        self.voll = 200.0
        self.vosp = 50.0
        self.M_beta = 1e4

        self.reserve_up_ratio = 0.05
        self.reserve_dn_ratio = 0.02
        self.rt_up_ratio = 3.0
        self.rt_dn_ratio = 0.5

        self.device = "cuda"
        self.epochs = 1
        self.dfl_train_batch_size = 8
        self.dfl_test_batch_size = 32
        self.batch_size = 4
        self.lr = 1e-8
        self.solver = "ECOS"

        self.N_scen = 20       # <== OptNet真正求解的场景池 (即 K)
        self.S_full = 100       # VAE 现场吐出的大量候选场景数 (S 池)
        self.K_rand = 10       # K里面有多少条纯随机保留(防过拟合)
        self.tau_gumbel = 0.1     # Gumbel Softmax 温度
        self.eps_uniform = 0.1 # 防震荡平滑参数
        self.lambda_div = 1e5   # [新增] 避免多头选到同一个场景的相互排斥惩罚力度

        self.sample_chunk = 10
        self.cpu_offload = False          # 测试想开再改 True（训练必须 False）
        self.shared_noise = True

        self.diffusion_mode = "ddpm"                # "ddpm" or "ddim"
        self.n_steps = 50                 # 仅 DDIM 用
        self.eta = 0.0                    # 仅 DDIM 用

        self.trunc_steps = 1              # 训练建议 1~3；测试建议 0（全程不需要可导）
        self.filter_epochs = 5 # Stage 2 (训Filter) 轮数
        self.filter_lr = 1e-3   # Stage 2 学习率
        self.dfl_epochs = 1     # Stage 3 (联合微调) 轮数 (端到端微调极耗时，一般1-3轮即收敛)
        self.dfl_lr = 1e-6      # Stage 3 学习率 (必须极小，防崩坏)

args = Args()

In [ ]:
set_seed(42)
DTYPE = torch.float64
data_path = "../Data/load_data_city_4_2.csv"
target_nodes = [f"4-2-{i}" for i in range(11)]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data=pd.read_csv(data_path)
data["DATETIME"] = pd.to_datetime(data["DATETIME"], errors="coerce")
data_2022 = data[data["DATETIME"].dt.year == 2022].copy()
Lmin, Lmax = system_hourly_load_minmax(data_2022, datetime_col="DATETIME",node_cols=target_nodes)
Lmax_total=Lmax.sum(0)# (24,)
Lmin_total=Lmin.sum(0) # (24,)

eps_search=pd.read_csv('../Result/eps_search.csv')
eps=int(eps_search[eps_search['model']=='diffusion_m']['eps'])
args.eps_value=eps

In [ ]:
data_path = "load_data_city_4_2.csv"
target_nodes = [f"4-2-{i}" for i in range(11)]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(42)
# =========================================================
# 1) train diffusion joint + tune z_temp on val
# =========================================================
models_m, handlers_m, pack_data_m = run_diffusion_joint(
    data_path=data_path,
    node_cols=target_nodes,
    device=device,
    epochs=500,
    batch_size=128,
    lr=1e-3,
    patience=50,
    ckpt_dir="../Model/Diffusion/ckpt_nodes_diffusion_joint_unet",
    verbose=True,
    train_length=8760,
    val_ratio=0.2,
    seed=42,
    # diffusion params
    T=100,
    time_dim=128,
    hidden_ch=256,
    n_layers=3,
    dropout=0.0,
    # z_temp tuning
    find_best_ztemp=True,
    z_temp_grid=(0.5, 0.75, 1.0, 1.25,1.5),
    ztemp_search_quantiles=[0.05 * i for i in range(1, 20)],
    ztemp_search_n_samples=50,
    ztemp_search_batch_size=64,
    ztemp_search_sample_chunk=20,
    ztemp_search_mode="ddpm",
    ztemp_search_n_steps=50,
    ztemp_search_eta=0.0,
    ztemp_search_shared_noise=True,
)

print("best_z_temp =", pack_data_m["_JOINT_"]["best_z_temp"])
print("best_val_pinball_real =", pack_data_m["_JOINT_"]["best_val_pinball_real"])
print("ztemp_table =", pack_data_m["_JOINT_"]["ztemp_table"])


Epoch    1 | train=1.073111 | val=0.897312
Epoch   10 | train=0.468498 | val=0.439363
Epoch   20 | train=0.279031 | val=0.294942
Epoch   30 | train=0.218151 | val=0.261110
Epoch   40 | train=0.209802 | val=0.296714
Epoch   50 | train=0.200056 | val=0.277786
Epoch   60 | train=0.197808 | val=0.276729
Epoch   70 | train=0.180114 | val=0.189982
Epoch   80 | train=0.176479 | val=0.214876
Epoch   90 | train=0.171785 | val=0.210771
Epoch  100 | train=0.180244 | val=0.258709
Epoch  110 | train=0.189425 | val=0.282419
Epoch  120 | train=0.182372 | val=0.206740
Epoch  130 | train=0.145201 | val=0.252455
Epoch  140 | train=0.167543 | val=0.208010
Epoch  150 | train=0.178589 | val=0.201849
Epoch  160 | train=0.177580 | val=0.211219
Epoch  170 | train=0.163027 | val=0.230738
Epoch  180 | train=0.172780 | val=0.217426
Epoch  190 | train=0.156712 | val=0.221242
Epoch  200 | train=0.154121 | val=0.209131
Epoch  210 | train=0.157274 | val=0.210560
Epoch  220 | train=0.145268 | val=0.217964
Epoch  230 

In [39]:

set_seed(0)
window_pack_diff_m_train = sample_window_diffusion_joint(
    models_m, handlers_m, pack_data_m,
    target_nodes=target_nodes,
    horizon_days=292, 
    start_day=0,
    n_samples=200,
    seq_len=24,
    split="train",
    mode="ddpm",
    n_steps=50,
    eta=0.0,
)

set_seed(0)
window_pack_diff_m_val = sample_window_diffusion_joint(
    models_m, handlers_m, pack_data_m,
    target_nodes=target_nodes,
    horizon_days=73, 
    start_day=0,
    n_samples=200,
    seq_len=24,
    split="val",
    mode="ddpm",
    n_steps=50,
    eta=0.0,
)

set_seed(0)
window_pack_diff_m_test = sample_window_diffusion_joint(
    models_m, handlers_m, pack_data_m,
    target_nodes=target_nodes,
    horizon_days=303, 
    start_day=0,
    n_samples=200,
    seq_len=24,
    split="test",
    mode="ddpm",
    eta=0.0,
)

In [40]:
import pickle
with open('../Result/Diffusion/models_m.pkl', 'wb') as f:
    pickle.dump(models_m, f)
with open('../Result/Diffusion/handlers_m.pkl', 'wb') as f:
    pickle.dump(handlers_m, f)
with open('../Result/Diffusion/pack_data_m.pkl', 'wb') as f:
    pickle.dump(pack_data_m, f)

In [41]:
import pickle
with open('../Result/Diffusion/window_pack_m_diff_train.pkl', 'wb') as f:
    pickle.dump(window_pack_diff_m_train, f)
with open('../Result/Diffusion/window_pack_m_diff_test.pkl', 'wb') as f:
    pickle.dump(window_pack_diff_m_test, f)
with open('../Result/Diffusion/window_pack_m_diff_val.pkl', 'wb') as f:
    pickle.dump(window_pack_diff_m_val, f)